# WMDP Target-Model Corrupt-Hook Mini GPU Test

This notebook is the next gate after the target-model full no-corrupt WMDP baseline. It loads the real target model, runs a baseline mini evaluation, then wraps the same model with `AttackedModel` and runs deterministic corruption methods on WMDP bio/chem/cyber with `sample_size=2`.

This is intentionally classifier-free: `prompt_classifier=None` attacks all WMDP prompts. Use this to validate the corruption hook path before running classifier-gated PoRT experiments with `WMDP_CLASSIFIER_PATH`.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys
import time

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()

def has_project_layout(path):
    path = Path(path)
    return (path / 'llm-unlearn-eco' / 'eco').exists() and (path / 'dataset').exists()

def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()

PROJECT_ROOT = clone_or_use_project()
ECO_ROOT = PROJECT_ROOT / 'llm-unlearn-eco'
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
if str(ECO_ROOT) not in sys.path:
    sys.path.insert(0, str(ECO_ROOT))

commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')


In [ ]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'tabulate': 'tabulate',
    'tqdm': 'tqdm',
    'transformers': 'transformers>=4.38.0',
    'accelerate': 'accelerate>=0.26.0',
    'yaml': 'pyyaml',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


In [ ]:
import gc
import platform
import pandas as pd
import torch
import yaml

from eco.attack import AttackedModel
from eco.attack.utils import remove_hooks
from eco.dataset import WMDPBio, WMDPChem, WMDPCyber
from eco.evaluator import ChoiceByTopLogit
from eco.inference import EvaluationEngine
from eco.model import HFModel
from eco.paths import MODEL_CONFIG_DIR

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is required for this corrupt-hook mini test. Enable a Kaggle GPU runtime and rerun.')

for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f'GPU {idx}: {props.name}, VRAM={props.total_memory / 1024**3:.2f} GiB')


## Runtime Model And Corruption Config


In [ ]:
TARGET_CONFIG_NAME = os.environ.get('PORT_TARGET_CONFIG_NAME', 'phi-1_5')
TARGET_HF_NAME = os.environ.get('PORT_TARGET_HF_NAME')
TARGET_MODEL_PATH = os.environ.get('PORT_TARGET_MODEL_PATH') or None
ATTN_IMPLEMENTATION = os.environ.get('PORT_ATTN_IMPLEMENTATION', 'eager')
TORCH_DTYPE = os.environ.get('PORT_TORCH_DTYPE', 'float16')
BATCH_SIZE = int(os.environ.get('PORT_WMDP_BATCH_SIZE', '1'))
SAMPLE_SIZE = int(os.environ.get('PORT_WMDP_SAMPLE_SIZE', '2'))
CORRUPT_METHODS = [m.strip() for m in os.environ.get('PORT_CORRUPT_METHODS', 'zero_out_first_n,flip_sign_first_n').split(',') if m.strip()]
CORRUPT_STRENGTH = float(os.environ.get('PORT_CORRUPT_STRENGTH', '1000.0'))

base_config_path = MODEL_CONFIG_DIR / f'{TARGET_CONFIG_NAME}.yaml'
if not base_config_path.exists():
    raise FileNotFoundError(f'Model config not found: {base_config_path}')

with open(base_config_path, 'r', encoding='utf-8') as f:
    runtime_config = yaml.safe_load(f)

if TARGET_HF_NAME:
    runtime_config['hf_name'] = TARGET_HF_NAME
runtime_config['attn_implementation'] = ATTN_IMPLEMENTATION
runtime_config['torch_dtype'] = TORCH_DTYPE
runtime_config['load_in_8bit'] = bool(runtime_config.get('load_in_8bit', False))
runtime_config['load_in_4bit'] = bool(runtime_config.get('load_in_4bit', False))
embedding_dim = int(runtime_config['embedding_dim'])
CORRUPT_DIMS = int(os.environ.get('PORT_CORRUPT_DIMS', str(embedding_dim)))

RUN_NAME = f'wmdp_target_corrupt_hook_mini_{TARGET_CONFIG_NAME}'
RUN_DIR = (Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT / 'results') / RUN_NAME
RUNTIME_CONFIG_DIR = RUN_DIR / 'model_config'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_MODEL_NAME = f'{TARGET_CONFIG_NAME}_runtime_gpu_corrupt_hook_mini'
runtime_config_path = RUNTIME_CONFIG_DIR / f'{RUNTIME_MODEL_NAME}.yaml'
with open(runtime_config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(runtime_config, f, sort_keys=False)

run_config = {
    'repo_url': REPO_URL,
    'commit_sha': commit_sha,
    'target_config_name': TARGET_CONFIG_NAME,
    'target_hf_name': runtime_config.get('hf_name'),
    'target_model_path': TARGET_MODEL_PATH,
    'runtime_model_name': RUNTIME_MODEL_NAME,
    'runtime_config_path': str(runtime_config_path),
    'attn_implementation': ATTN_IMPLEMENTATION,
    'torch_dtype': TORCH_DTYPE,
    'batch_size': BATCH_SIZE,
    'sample_size': SAMPLE_SIZE,
    'corrupt_methods': CORRUPT_METHODS,
    'corrupt_dims': CORRUPT_DIMS,
    'corrupt_strength': CORRUPT_STRENGTH,
    'classifier_mode': 'none_attack_all_wmdp_prompts',
    'datasets': ['wmdp-bio', 'wmdp-chem', 'wmdp-cyber'],
    'run_dir': str(RUN_DIR),
}
(RUN_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
print(json.dumps(run_config, indent=2))


## Load Target Model


In [ ]:
start_model = time.perf_counter()
base_model = HFModel(
    model_name=RUNTIME_MODEL_NAME,
    model_path=TARGET_MODEL_PATH,
    config_path=str(RUNTIME_CONFIG_DIR),
)
if base_model.tokenizer.pad_token_id is None:
    base_model.tokenizer.pad_token = base_model.tokenizer.eos_token
base_model.tokenizer.padding_side = 'right'
model_load_seconds = time.perf_counter() - start_model
print(f'Model loaded in {model_load_seconds:.2f}s')
if hasattr(base_model.model, 'hf_device_map'):
    print('HF device map:', json.dumps(base_model.model.hf_device_map, default=str, indent=2))
print('CUDA memory allocated GiB:', torch.cuda.memory_allocated() / 1024**3)
print('CUDA memory reserved GiB:', torch.cuda.memory_reserved() / 1024**3)


## Run Baseline And Corrupt-Hook Mini Evaluations


In [ ]:
choice_labels = ['A', 'B', 'C', 'D']
strength_methods = {
    'rand_noise_first_n',
    'rand_noise_rand_n',
    'rand_noise_top_k',
    'sub_value_top_k',
    'add_value_least_k',
    'set_rand_noise_first_n',
    'sub_value_first_n',
    'add_value_first_n',
}
all_summaries = []
all_predictions = []
timings = {}


def build_corrupt_args(method):
    args = {'dims': CORRUPT_DIMS}
    if method in strength_methods:
        args['strength'] = CORRUPT_STRENGTH
    return args


def run_eval(run_label, eval_model, corrupt_method=None, corrupt_args=None):
    run_start = time.perf_counter()
    run_predictions = []
    run_summaries = []
    for data_module in [WMDPBio(sample_size=SAMPLE_SIZE), WMDPChem(sample_size=SAMPLE_SIZE), WMDPCyber(sample_size=SAMPLE_SIZE)]:
        print(f'\nRunning {run_label} on {data_module.name}...')
        subject_start = time.perf_counter()
        engine = EvaluationEngine(
            model=eval_model,
            tokenizer=eval_model.tokenizer,
            data_module=data_module,
            subset_names=['test'],
            evaluator=ChoiceByTopLogit(),
            batch_size=BATCH_SIZE,
        )
        engine.inference()
        summary_stats, outputs = engine.summary()
        elapsed = time.perf_counter() - subject_start
        run_summaries.extend(summary_stats)

        result_name, batches = list(outputs[0].items())[0]
        row_idx = 0
        for batch in batches:
            for c, p in zip(batch['correct'], batch['predicted']):
                row = {
                    'run_label': run_label,
                    'dataset': data_module.name,
                    'row_index': row_idx,
                    'correct_index': int(c),
                    'predicted_index': int(p),
                    'correct_label': choice_labels[int(c)] if 0 <= int(c) < len(choice_labels) else str(c),
                    'predicted_label': choice_labels[int(p)] if 0 <= int(p) < len(choice_labels) else str(p),
                    'is_correct': int(c) == int(p),
                    'corrupt_method': corrupt_method,
                    'corrupt_args': json.dumps(corrupt_args, default=str) if corrupt_args else None,
                }
                run_predictions.append(row)
                all_predictions.append(row)
                row_idx += 1
        print(f'{run_label}/{data_module.name} completed in {elapsed:.2f}s with {row_idx} rows.')
    timings[run_label] = time.perf_counter() - run_start
    all_summaries.append({'run_label': run_label, 'summaries': run_summaries})
    pd.DataFrame(run_predictions).to_csv(RUN_DIR / f'predictions_{run_label}.csv', index=False)


run_eval('baseline_none', base_model)
for method in CORRUPT_METHODS:
    remove_hooks(base_model.model)
    corrupt_args = build_corrupt_args(method)
    attacked_model = AttackedModel(
        model=base_model,
        prompt_classifier=None,
        token_classifier=None,
        corrupt_method=method,
        corrupt_args=corrupt_args,
        classifier_threshold=0.0,
    )
    run_eval(f'corrupt_{method}', attacked_model, corrupt_method=method, corrupt_args=corrupt_args)
    remove_hooks(base_model.model)

predictions_df = pd.DataFrame(all_predictions)
summary_by_run = predictions_df.groupby(['run_label', 'dataset'])['is_correct'].agg(['count', 'mean']).reset_index()
summary_by_run = summary_by_run.rename(columns={'mean': 'accuracy'})
print(summary_by_run)


In [ ]:
summary_payload = {
    'run_config': run_config,
    'model_load_seconds': model_load_seconds,
    'timings_seconds': timings,
    'engine_summaries': all_summaries,
    'summary_by_run': summary_by_run.to_dict(orient='records'),
    'total_rows': int(len(predictions_df)),
}
summary_path = RUN_DIR / 'summary.json'
predictions_path = RUN_DIR / 'predictions.csv'
summary_by_run_path = RUN_DIR / 'summary_by_run.csv'
summary_path.write_text(json.dumps(summary_payload, indent=2, default=str), encoding='utf-8')
predictions_df.to_csv(predictions_path, index=False)
summary_by_run.to_csv(summary_by_run_path, index=False)
print(f'Wrote {summary_path}')
print(f'Wrote {predictions_path}')
print(f'Wrote {summary_by_run_path}')
print(json.dumps(summary_payload, indent=2, default=str)[:4000])


In [ ]:
del base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('WMDP TARGET-MODEL CORRUPT-HOOK MINI GPU TEST COMPLETED')
